# B6 Stage1 — Time Structure Discovery with 5 Fixed SLs / TPなし

**Workでは全探索未実行。ChatがFreeze内容とremote SHAを確認した後、Google Colabで実行します。**
既定の全セル実行は設定・探索空間の表示だけです。Validation / Monitor実行セルはありません。
Notebook内でSL・ranking・期間・曜日・探索刻みを変更しないでください。入力パスと確認済みSHAのみ設定します。

1. Chatが承認した40桁SHAをFREEZE_SHAへ設定。
2. PREPARE_ENVIRONMENT=Trueで固定コードと依存ライブラリを準備。
3. 必要ならDriveを**入力読込のため**mount、DATA_ROOTを設定。Driveへの成果物保存はOFF。
4. Chat確認済みの場合だけCHAT_CONFIRMED_FREEZE=True、RUN_FULL_SWEEP=True。
5. `/content/b6_stage1` の圧縮shard/SQLite/選定結果を保存。中断再開には同じコード・config・入力・出力一式が必要。

Colabランタイム破棄で/contentは消えます。再開用に必要なら出力フォルダ全体を利用者が退避してください。自動Drive保存はしません。


In [ ]:
from pathlib import Path
import json, sys, subprocess, os
sys.dont_write_bytecode = True
FREEZE_SHA = ''  # Chat確認済みGitHub commitの40桁SHA
PREPARE_ENVIRONMENT = False
RUN_FULL_SWEEP = False
CHAT_CONFIRMED_FREEZE = False
MOUNT_DRIVE_FOR_INPUTS = False
SAVE_TO_DRIVE = False
DATA_ROOT = Path('/content/drive/MyDrive/<M1_ROOT>')
REPO = Path('/content/b6_stage1_repo')
OUT = Path('/content/b6_stage1')


In [ ]:
DISPLAY_CONFIG = {'schema': 'b6-stage1-v1', 'status': 'APPROVED_STAGE1_IMPLEMENTATION_FREEZE', 'decision_date': '2026-09-28', 'discovery_start': '2020-01-01', 'discovery_end_exclusive': '2024-01-01', 'symbols': ['USDJPY', 'EURJPY', 'GBPJPY', 'AUDJPY', 'AUDUSD', 'EURAUD', 'GBPAUD'], 'directions': ['L', 'S'], 'weekdays': [0, 1, 2, 3, 4], 'entry_step_minutes': 5, 'holding_minutes': {'min': 30, 'max': 1440, 'step': 5}, 'sl_pips': {'USDJPY': [10, 25, 40, 60, 95], 'EURJPY': [15, 30, 50, 75, 115], 'GBPJPY': [20, 40, 65, 100, 150], 'AUDJPY': [10, 25, 40, 65, 100], 'AUDUSD': [10, 20, 35, 55, 85], 'EURAUD': [20, 35, 60, 95, 145], 'GBPAUD': [20, 40, 70, 110, 160]}, 'tp_pips': None, 'entry_year_end_stop': {'from': '12-25', 'through': '01-03'}, 'spread_pips': {'USDJPY': 0.5, 'EURJPY': 1, 'GBPJPY': 2, 'AUDJPY': 1.5, 'AUDUSD': 1.5, 'EURAUD': 1.5, 'GBPAUD': 2}, 'gate': {'min_trades': 150, 'min_annual_trades': 30, 'min_losses': 10, 'min_pf': 1.1, 'min_positive_years': 3, 'min_passing_sl': 3}, 'ranking': ['median_all5_AvgR_DESC', 'PassSLCount_DESC', 'median_all5_TotalR_DESC', 'worst_all5_MaxDDR_ASC', 'symbol_ASC', 'direction_L_before_S', 'weekday_ASC', 'entry_minute_ASC', 'exit_day_offset_ASC', 'exit_minute_ASC'], 'clustering': {'radius_entry_minutes': 30, 'radius_exit_minutes': 30, 'same_day_offset': True, 'max_hold_difference_minutes': 30, 'algorithm': 'rank_order_direct_representative_no_transitive_merge', 'max_structures': 50}, 'metrics': {'basis': 'unrounded_R', 'annual_basis': 'JST_entry_year', 'dd_order': 'close_then_entry', 'dd_initial_peak_R': 0, 'display_pips_decimals': 6, 'display_R_decimals': 9}, 'execution': {'timezone': 'Europe/Helsinki->Asia/Tokyo naive', 'ambiguous': 'infer', 'nonexistent': 'shift_forward', 'duplicates': 'reject', 'entry_exact': True, 'entry_bar_inclusive': True, 'exit_bar_inclusive': True, 'same_bar': 'SL_FIRST', 'max_exit_fallback_minutes': 4, 'ensure_exit_before_path': True, 'hit_epsilon': 0, 'interpolation': False, 'gap_fill': 'fixed_SL_price'}, 'full_sweep_environment': 'Google Colab after Chat confirmation', 'validation_enabled': False, 'monitor_enabled': False, 'output': {'default_root': '/content/b6_stage1', 'drive_save_default': False, 'all5_metrics': 'compressed_npz_shards', 'ranking': 'sqlite_external_sort', 'assignments': 'csv.gz', 'raw_M1_saved': False, 'all_trade_log_saved': False}}
h=DISPLAY_CONFIG['holding_minutes']
structures=len(DISPLAY_CONFIG['symbols'])*2*5*len(range(0,1440,5))*len(range(h['min'],h['max']+1,h['step']))
print('期間: Discovery 2020-2023 / TPなし')
print('時間構造:',f'{structures:,}','SL設定:',f'{structures*5:,}')
print('Entry/Exit: 5分、予定保有30〜1440分、Entry曜日Mon〜Friを個別探索')
print('ペア別SL pips:')
for pair,sl in DISPLAY_CONFIG['sl_pips'].items(): print(pair,sl)
print('最低条件:',DISPLAY_CONFIG['gate'])
print('順位:',DISPLAY_CONFIG['ranking'])
print('近接整理:',DISPLAY_CONFIG['clustering'])
print('これは設定表示であり、Candidate成績ではありません。')


In [ ]:
if PREPARE_ENVIRONMENT:
    assert len(FREEZE_SHA)==40 and all(c in '0123456789abcdef' for c in FREEZE_SHA)
    try:
        import google.colab
    except ImportError:
        raise RuntimeError('実データfull sweepの準備はGoogle Colabで行ってください')
    subprocess.run([sys.executable,'-m','pip','install','--quiet','numpy==2.3.5','pandas==2.2.3'],check=True)
    if not REPO.exists():
        subprocess.run(['git','clone','https://github.com/TR-KJ/time-entry-portfolio-lab.git',str(REPO)],check=True)
    subprocess.run(['git','diff','--quiet','HEAD','--'],cwd=REPO,check=True)
    subprocess.run(['git','fetch','origin',FREEZE_SHA],cwd=REPO,check=True)
    subprocess.run(['git','checkout','--detach',FREEZE_SHA],cwd=REPO,check=True)
    sys.path.insert(0,str(REPO/'src/research'))
    import numpy, pandas
    assert numpy.__version__=='2.3.5' and pandas.__version__=='2.2.3', 'ランタイムを再起動して同じSHAでやり直してください'
    from b6.stage1_search import verify_release
    verify_release(FREEZE_SHA)
    from b6.stage1_config import load_config,search_space
    assert load_config()==DISPLAY_CONFIG, 'Notebookと固定configが異なります'
    print('固定source:',FREEZE_SHA)
    print(search_space())
else:
    print('環境準備OFF。ネットワーク接続・データ読込なし。')


In [ ]:
if MOUNT_DRIVE_FOR_INPUTS:
    assert PREPARE_ENVIRONMENT
    from google.colab import drive
    drive.mount('/content/drive')
assert not SAVE_TO_DRIVE, 'このNotebookはDriveへの自動出力を実装していません'


## Stage1 full sweep — 明示操作後だけ実行

4,032 job。各job内で5曜日×283保有×5SLを評価します。全job完了後だけranking/直接近接整理/最大50構造選定を出力します。
失格SLを含め5SLの全指標を保持します。実行時間・圧縮後容量の全探索実測はまだありません。
入力監査→Discovery配列作成→SL到達検索→全保有時間の指標→外部sort→最大50構造、の順です。


In [ ]:
if RUN_FULL_SWEEP:
    assert PREPARE_ENVIRONMENT and CHAT_CONFIRMED_FREEZE
    assert DATA_ROOT.exists() and str(OUT)=='/content/b6_stage1'
    from b6.stage1_search import full_sweep
    full_sweep(DATA_ROOT,OUT,FREEZE_SHA,confirmed=CHAT_CONFIRMED_FREEZE)
else:
    print('Stage1 FULL SWEEP: NOT RUN')


In [ ]:
if RUN_FULL_SWEEP:
    report=json.loads((OUT/'progress.json').read_text())
    assert report['State']=='COMPLETE_STAGE1_ONLY'
    from IPython.display import display
    import pandas as pd
    print('試行数:',json.loads((OUT/'search_space.json').read_text()))
    print('絞込み:',json.loads((OUT/'selection_summary.json').read_text()))
    selected=json.loads((OUT/'stage1_selected_structures.json').read_text())
    display(pd.DataFrame([{k:v for k,v in r.items() if k!='SLResults'} for r in selected]))
    for r in selected:
        print(r['CandidateID'],'通過SL数',r['PassSLCount'],'/ 5')
        display(pd.DataFrame(r['SLResults']))  # 年別N/PF/AvgR/TotalR/DDも表示
    print('選定理由・抑制先: assignments.csv.gz。全5SL指標: shards/*.npz。')
    print('Stage2/Validation/Monitor: NOT RUN')


## 出力と境界

- `identity.json`：コードSHA・config hash・56入力hash。
- `input_audit.csv`、`search_space.json`、`effective_config.json`：入力・正式条件・試行数。
- `shards/*.npz`：全5SL、通過/不通過、年別指標、missing/filtered/fallback診断。
- `stage1.sqlite`：通過時間構造とjob完了記録。
- `assignments.csv.gz`：全通過時間構造の順位、代表/抑制/50枠外、代表との距離。
- `stage1_selected_structures.json`：最大50時間構造と全5SL。Stage2へ渡す候補でありlive採用ではありません。

出力のlarge CSV/SQLite/生データをGitHubへ大量投入しないでください。Stage2以降は別指示・別の実行コードで扱います。
